# Phase 2 — Gold Layer: Analytical Aggregates

Three aggregates built from `silver.orders`: daily revenue by category
(item grain), top 10 customers by lifetime value, and a 7-day rolling
average of order volume. The latter two are computed on a deduplicated
order-grain view — the Silver table has one row per item, so summing
`order_amount` or counting `order_id` directly would double count orders
with multiple items.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load Silver base table

In [ ]:
silver_orders = spark.table("silver.orders")

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

## Aggregate 1 — Daily revenue by product category

Revenue per day per category, summed at item grain (`price +
freight_value`).

In [ ]:
daily_revenue_by_category = (
    silver_orders
    .groupBy("order_date", "product_category_name_english")
    .agg(F.sum(F.col("price") + F.col("freight_value")).alias("revenue"))
    .orderBy("order_date", "product_category_name_english")
)

## Write gold.daily_revenue_by_category

In [ ]:
daily_revenue_by_category.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gold.daily_revenue_by_category")

print(
    f"gold.daily_revenue_by_category: {spark.table('gold.daily_revenue_by_category').count()} rows"
)

_ = spark.sql("OPTIMIZE gold.daily_revenue_by_category ZORDER BY (order_date)")

In [ ]:
display(spark.table("gold.daily_revenue_by_category").limit(10))

## Order-grain view for the remaining aggregates

Dedupe to one row per `order_id` before aggregating on order-level fields
(`order_amount`, `customer_unique_id`).

In [ ]:
orders_dedup = (
    silver_orders
    .dropDuplicates(["order_id"])
    .select("order_id", "customer_unique_id", "order_date", "order_amount")
)

## Aggregate 2 — Top 10 customers by lifetime value

LTV grouped by `customer_unique_id`, not `customer_id` — in this dataset
`customer_id` is effectively one-per-order, so `customer_unique_id` is
the real customer identity.

In [ ]:
top10_customers_ltv = (
    orders_dedup
    .groupBy("customer_unique_id")
    .agg(F.sum("order_amount").alias("lifetime_value"))
    .orderBy(F.desc("lifetime_value"))
    .limit(10)
)

## Write gold.top10_customers_ltv

No meaningful date column on a 10-row table, so `OPTIMIZE` is applied
without `ZORDER`.

In [ ]:
top10_customers_ltv.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gold.top10_customers_ltv")

print(
    f"gold.top10_customers_ltv: {spark.table('gold.top10_customers_ltv').count()} rows"
)

_ = spark.sql("OPTIMIZE gold.top10_customers_ltv")

In [ ]:
display(spark.table("gold.top10_customers_ltv"))

## Aggregate 3 — 7-day rolling average order volume

Daily distinct order count, then a trailing 7-day (`rowsBetween(-6, 0)`)
average ordered by date.

In [ ]:
daily_order_volume = (
    orders_dedup
    .groupBy("order_date")
    .agg(F.countDistinct("order_id").alias("order_count"))
)

rolling_window = Window.orderBy("order_date").rowsBetween(-6, 0)

rolling_7day_order_volume = daily_order_volume.withColumn(
    "rolling_7day_avg_order_volume", F.avg("order_count").over(rolling_window)
).orderBy("order_date")

## Write gold.rolling_7day_order_volume

In [ ]:
rolling_7day_order_volume.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gold.rolling_7day_order_volume")

print(
    f"gold.rolling_7day_order_volume: {spark.table('gold.rolling_7day_order_volume').count()} rows"
)

_ = spark.sql("OPTIMIZE gold.rolling_7day_order_volume ZORDER BY (order_date)")

In [ ]:
display(spark.table("gold.rolling_7day_order_volume").limit(10))